# Lesson 03 — Lowe's Ratio Test: Eliminating Bad Matches

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img1 = cv2.imread('sample.jpg')
img2 = img1.copy()
M = cv2.getRotationMatrix2D((img1.shape[1]//2, img1.shape[0]//2), 25, 0.8)
img2 = cv2.warpAffine(img2, M, (img2.shape[1], img2.shape[0]))

sift = cv2.SIFT_create(nfeatures=500)
kp1, d1 = sift.detectAndCompute(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), None)
kp2, d2 = sift.detectAndCompute(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), None)

bf      = cv2.BFMatcher()
matches = bf.knnMatch(d1, d2, k=2)  # k=2: return 2 best matches

# All matches (many are wrong)
all_matches = [m for m,n in matches]

# Lowe's ratio test: good match only if best is much better than second-best
good_matches = [m for m,n in matches if m.distance < 0.75 * n.distance]

print(f"Total matches: {len(all_matches)}")
print(f"After ratio test (0.75): {len(good_matches)} — {len(good_matches)/len(all_matches)*100:.0f}% survived")

fig, axes = plt.subplots(1,2,figsize=(18,5))
vis_all  = cv2.drawMatches(img1,kp1,img2,kp2,all_matches[:80],  None,flags=2)
vis_good = cv2.drawMatches(img1,kp1,img2,kp2,good_matches[:50], None,flags=2)
axes[0].imshow(cv2.cvtColor(vis_all,  cv2.COLOR_BGR2RGB)); axes[0].set_title(f'All {len(all_matches)} matches — chaotic'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(vis_good, cv2.COLOR_BGR2RGB)); axes[1].set_title(f'{len(good_matches)} after ratio test — clean!'); axes[1].axis('off')
plt.show()

## Key Takeaway
Ratio test: keep match only if `best_distance < ratio * second_best_distance`.
ratio=0.75 is Lowe's original recommendation. Lower ratio = fewer but more reliable matches.
This single trick eliminates 80% of bad matches.